To do:

- Filter for fbs teams only
- Get rid of test/ train aspect (train on 2020-2024 and predict on given 2025 game id entry)
- validate the 2025 game id entry for the text box
- get visualizations and predictions as output on the webpage
- make webpage look nice

In [ ]:
# import library
import streamlit as st
import os
import tabulate
import requests
import pandas as pd
import numpy as np
import pyarrow
import sportsdataverse as sdv
import polars as pl
from great_tables import GT, md
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

---
Win probability model Code

In [ ]:
# Set seed for reproducibility
np.random.seed(36)

In [ ]:
CFBD_API_KEY = "cKf6FOVq1WGS2xOlvMoAMw6TB3ELUG7vX1h6FoQcqBceFCHHfJBDfs+WrpABR8xy"
headers = {"Authorization": f"Bearer {CFBD_API_KEY}"}

In [ ]:
seasons = range(2020, 2025) # changed to 2025 to include up to the 2024 season
weeks = range(1, 16)
raw_stats_list = []

# Data ingestion loop
for yr in seasons:
    for wk in weeks:
        url = f"https://api.collegefootballdata.com/games/teams?year={yr}&week={wk}&seasonType=regular&division=fbs"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                data = response.json()
                for game in data:
                    game_id = game.get("id")
                    teams = game.get("teams", [])
                    if len(teams) == 2:
                        for idx, team_data in enumerate(teams):
                            opp_data = teams[1 - idx]
                            
                            # Helper to extract metric by name
                            def get_stat(t_data, stat_name):
                                for s in t_data.get("stats", []):
                                    if s.get("category") == stat_name:
                                        return s.get("stat")
                                return None

                            row = {
                                "game_id": game_id,
                                "season": str(yr),
                                "week_num": wk,
                                "team": team_data.get("school"),
                                "opponent": opp_data.get("school"),
                                "home_away": "home" if team_data.get("homeAway") == "home" else "away",
                                "points": team_data.get("points"),
                                "points_allowed": opp_data.get("points"),
                                "total_yards": get_stat(team_data, "totalYards"),
                                "total_yards_allowed": get_stat(opp_data, "totalYards"),
                                "net_passing_yards": get_stat(team_data, "netPassingYards"),
                                "net_passing_yards_allowed": get_stat(opp_data, "netPassingYards"),
                                "rushing_yards": get_stat(team_data, "rushingYards"),
                                "rushing_yards_allowed": get_stat(opp_data, "rushingYards"),
                                "turnovers": get_stat(team_data, "turnovers"),
                                "turnovers_allowed": get_stat(opp_data, "turnovers"),
                                "first_downs": get_stat(team_data, "firstDowns"),
                                "first_downs_allowed": get_stat(opp_data, "firstDowns"),
                                "third_down_eff": get_stat(team_data, "thirdDownEff"),
                                "third_down_eff_allowed": get_stat(opp_data, "thirdDownEff"),
                                "total_penalties_yards": get_stat(team_data, "totalPenaltiesYards"),
                                "total_penalties_yards_allowed": get_stat(opp_data, "totalPenaltiesYards"),
                                "possession_time": get_stat(team_data, "possessionTime"),
                                "possession_time_allowed": get_stat(opp_data, "possessionTime"),
                            }
                            raw_stats_list.append(row)
        except Exception as e:
            continue

raw_team_stats = pd.DataFrame(raw_stats_list)

# Filter for both teams being FBS here.


In [ ]:
def parse_time_sec(time_str):
    if pd.isna(time_str):
        return 1800.0
    parts = str(time_str).split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    return 1800.0

def parse_split_eff(eff_str, pos=0):
    if pd.isna(eff_str):
        return 0.0
    parts = str(eff_str).split("-")
    if len(parts) == 2:
        try:
            return float(parts[pos])
        except ValueError:
            return 0.0
    return 0.0



In [ ]:
df = raw_team_stats.copy()
df = df.dropna(subset=["points", "points_allowed"])
df = df[df["points"] != df["points_allowed"]]

df["points"] = df["points"].astype(float)
df["points_allowed"] = df["points_allowed"].astype(float)
df["win"] = (df["points"] > df["points_allowed"]).astype(int)

# Parsing nested string fields
df["td_comp"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 0))
df["td_att"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 1))
df["td_comp_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 0))
df["td_att_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["pen_yds"] = df["total_penalties_yards"].apply(lambda x: parse_split_eff(x, 1))
df["pen_yds_opp"] = df["total_penalties_yards_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["top_sec"] = df["possession_time"].apply(parse_time_sec)
df["top_sec_opp"] = df["possession_time_allowed"].apply(parse_time_sec)

df["third_down_pct"] = np.where(df["td_att"] > 0, df["td_comp"] / df["td_att"], 0.0)
df["third_down_pct_opp"] = np.where(df["td_att_opp"] > 0, df["td_comp_opp"] / df["td_att_opp"], 0.0)

num_cols = ["turnovers", "turnovers_allowed", "net_passing_yards", "net_passing_yards_allowed",
            "rushing_yards", "rushing_yards_allowed", "first_downs", "first_downs_allowed"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

In [ ]:
# Computing differentials
df["turnover_margin"] = df["turnovers_allowed"] - df["turnovers"]
df["pass_yards_diff"] = df["net_passing_yards"] - df["net_passing_yards_allowed"]
df["rush_yards_diff"] = df["rushing_yards"] - df["rushing_yards_allowed"]
df["first_downs_diff"] = df["first_downs"] - df["first_downs_allowed"]
df["third_down_pct_diff"] = df["third_down_pct"] - df["third_down_pct_opp"]
df["penalty_yards_diff"] = df["pen_yds"] - df["pen_yds_opp"]
df["top_seconds_diff"] = df["top_sec"] - df["top_sec_opp"]
df["is_home"] = (df["home_away"].str.lower() == "home").astype(int)

# Filter every 2nd row to prevent double-counting full game perspectives
cfb_diff = df.iloc[1::2].reset_index(drop=True)

diff_predictors = [
    "is_home", "turnover_margin", "pass_yards_diff", "rush_yards_diff",
    "first_downs_diff", "third_down_pct_diff", "penalty_yards_diff", "top_seconds_diff"
]

cfb_diff = cfb_diff.dropna(subset=["win"] + diff_predictors)
cfb_diff.head()

In [ ]:
X_diff = cfb_diff[diff_predictors]
y_diff = cfb_diff["win"]
strata_diff = cfb_diff["season"]

X_train, X_test, y_train, y_test = train_test_split(
    X_diff, y_diff, test_size=0.20, random_state=2026, stratify=strata_diff
)

test_indices = X_test.index
test_seasons = cfb_diff.loc[test_indices, "season"]

# Standardizing features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=diff_predictors, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=diff_predictors, index=X_test.index)

# Add constant for statsmodels Logit
X_train_sm = sm.add_constant(X_train_scaled)
X_test_sm = sm.add_constant(X_test_scaled)


In [ ]:
# Fit Logistic Regression Model
logit_model_diff = sm.Logit(y_train, X_train_sm).fit(disp=False)

# Feature Importance Summary Table
summary_df = pd.DataFrame({
    "Predictor": logit_model_diff.params.index,
    "Std Beta": logit_model_diff.params.values,
    "Odds Ratio (1 SD)": np.exp(logit_model_diff.params.values),
    "z-statistic": logit_model_diff.tvalues.values,
    "p-value": logit_model_diff.pvalues.values
})
summary_df = summary_df[summary_df["Predictor"] != "const"].copy()
summary_df["Importance (|z|)"] = summary_df["z-statistic"].abs()
summary_df = summary_df.sort_values(by="Importance (|z|)", ascending=False)

print(summary_df.to_string(index=False))

Calculate the Win probability (need to change get rid of the test/train aspect).

In [ ]:
pred_probs_diff = logit_model_diff.predict(X_test_sm)
pred_class_diff = (pred_probs_diff >= 0.5).astype(int)

metrics_diff = pd.DataFrame({
    "Metric": ["Accuracy", "ROC AUC", "Log Loss", "Brier Score"],
    "Test Score": [
        accuracy_score(y_test, pred_class_diff),
        roc_auc_score(y_test, pred_probs_diff),
        log_loss(y_test, pred_probs_diff),
        brier_score_loss(y_test, pred_probs_diff)
    ]
})

print("Out-of-Sample Performance (Differentials Model):")
print(metrics_diff.to_string(index=False))

---
Web Page Code

In [ ]:
st.title("2025 College Football Win Probability Predictor")
st.write("Welcome to the 2025 College Football Win Probability Predictor!")

name = st.text_input("Please Enter a Game ID (e.g., 401331352):")
if name:
    st.write(f"Hi, {name}!") # Change out with the model predictions and add visualizations for the game ID entered. Verify 2025 id entered.
